# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Волошин Михайло Сергійович  
**Група:** 4-9  
**Дата:** 23.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print("Кількість унікальних користувачів:", ratings_df['user_id'].nunique())
print("Кількість унікальних фільмів:", ratings_df['item_id'].nunique())
print(f"Мінімальна оцінка: {ratings_df['rating'].min()}")
print(f"Максимальна оцінка: {ratings_df['rating'].max()}")
print(f"Середня оцінка: {ratings_df['rating'].mean():.2f}")

print("\nПропущені значення:")
print(ratings_df.isnull().sum())

print("\nРозподіл оцінок:")
print(ratings_df['rating'].value_counts().sort_index())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53

Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір даних ml-100k містить 100 000 оцінок від 943 унікальних користувачів для 1 682 унікальних фільмів, пропущені значення відсутні. Оцінки представлені цілими числами в діапазоні від 1 до 5 із середнім значенням близько 3.53, що свідчить про помірний зсув розподілу в бік позитивних відгуків (найчастіше користувачі ставлять 4 та 3)

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print("Розмір train_df:", train_df.shape)
print("Розмір test_df:", test_df.shape)

Розмір train_df: (80000, 4)
Розмір test_df: (20000, 4)


In [ ]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print("Кількість записів у train_data:", len(train_data.raw_ratings))
print("Кількість записів у testset:", len(testset))

Кількість записів у train_data: 80000
Кількість записів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD обрано три ключові гіперпараметри: n_factors (50, 100) задає розмірність латентного простору (баланс між точністю та перенавчанням), lr_all (0.005, 0.010) контролює крок градієнтного спуску для стабільної збіжності, а reg_all (0.02, 0.05) додає регуляризацію для зменшення перенавчання на розрідженій матриці оцінок.

Для KNNBasic підбираються k (20, 40) — максимальна кількість найближчих сусідів, min_k (1, 5) — мінімальна кількість сусідів для формування прогнозу, а також sim_options: метрика схожості (cosine та pearson) і підхід (user_based: True — user-based CF, False — item-based CF), що дозволяє порівняти пошук схожих користувачів і схожих фільмів.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=5,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [ ]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=5,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [ ]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print("=== SVD ===")
print(f"Найкращий CV RMSE: {grid_search_svd.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE:  {grid_search_svd.best_score['mae']:.4f}")
print("Найкращі параметри за RMSE:", grid_search_svd.best_params['rmse'])

print("\n=== KNNBasic ===")
print(f"Найкращий CV RMSE: {grid_search_knn.best_score['rmse']:.4f}")
print(f"Найкращий CV MAE:  {grid_search_knn.best_score['mae']:.4f}")
print("Найкращі параметри за RMSE:", grid_search_knn.best_params['rmse'])

=== SVD ===
Найкращий CV RMSE: 0.9336
Найкращий CV MAE:  0.7358
Найкращі параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

=== KNNBasic ===
Найкращий CV RMSE: 1.0225
Найкращий CV MAE:  0.8109
Найкращі параметри за RMSE: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'pearson', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9336 | 0.7358 | n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05 |
| KNNBasic | 1.0225 | 0.8109 | 'k': 40, 'min_k': 5, 'sim_options': {'name': 'pearson', 'user_based': True |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._
За результатами 5-кратної кросвалідації модель SVD показала нижчі значення помилок RMSE та MAE порівняно з KNNBasic, оскільки матрична факторизація краще виявляє приховані залежності в розріджених даних. Найкращі результати для обох моделей досягнуто за параметрів, наведених у таблиці вище.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
# TODO 6.1
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']
# TODO 6.2
trainset = train_data.build_full_trainset()
# TODO 6.3
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print("Обидві моделі успішно навчено на повній навчальній вибірці.")

Computing the pearson similarity matrix...
Done computing similarity matrix.
Обидві моделі успішно навчено на повній навчальній вибірці.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
# TODO 7.1
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)
# TODO 7.2
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f"SVD      -> Test RMSE: {svd_rmse:.4f}, Test MAE: {svd_mae:.4f}")
print(f"KNNBasic -> Test RMSE: {knn_rmse:.4f}, Test MAE: {knn_mae:.4f}")

SVD      -> Test RMSE: 0.9192, Test MAE: 0.7234
KNNBasic -> Test RMSE: 1.0065, Test MAE: 0.7986


In [ ]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
}).sort_values(by='RMSE').reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.006452,0.798564


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0065 | 0.7986 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Порівняння з CV: Значення RMSE та MAE на незалежній тестовій вибірці дуже близькі до результатів 5-кратної кросвалідації (різниця мінімальна). Це свідчить про відсутність значного перенавчання та коректну узагальнювальну здатність обох моделей, причому лідерство моделі з кращим CV RMSE збереглося і на тесті.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
user_id = "196"

# TODO 8.1
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = "SVD"
else:
    best_model = best_knn
    best_model_name = "KNNBasic"

print(f"Обрана модель для рекомендацій (за CV RMSE): {best_model_name}")
# TODO 8.2
try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print(f"Користувача з user_id='{user_id}' немає в навчальній вибірці (проблема холодного старту).")
else:
# TODO 8.3
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    all_inner_iids = set(trainset.all_items())
    unrated_inner_iids = all_inner_iids - rated_inner_iids
# TODO 8.4
    predictions = []
    for inner_iid in unrated_inner_iids:
        raw_iid = trainset.to_raw_iid(inner_iid)
        est_rating = best_model.predict(uid=user_id, iid=raw_iid).est
        predictions.append((raw_iid, est_rating))

    top_10_df = (
        pd.DataFrame(predictions, columns=['item_id', 'predicted_rating'])
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
    display(top_10_df)

Обрана модель для рекомендацій (за CV RMSE): SVD

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

У ході лабораторної роботи на наборі даних ml-100k було побудовано та порівняно дві рекомендаційні моделі — SVD та KNNBasic. За результатами 5-кратної кросвалідації на навчальній вибірці обрано модель SVD, оскільки вона продемонструвала нижчий CV RMSE (0.9336 проти 1.0205 у KNNBasic). Найкращими гіперпараметрами за RMSE стали {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} для SVD та {'k': 40, 'min_k': 5, 'sim_options': {'name': 'pearson', 'user_based': True}} для KNNBasic.   На незалежній тестовій вибірці отримано такі підсумкові метрики: для SVD — RMSE = 0.9192, MAE = 0.7234; для KNNBasic — RMSE = 1.0065, MAE = 0.7986. Отримані тестові показники навіть трохи кращі за оцінки кросвалідації (різниця становить близько 0.014), що підтверджує стабільність моделей і відсутність перенавчання. За допомогою найкращої моделі (SVD) сформовано топ-10 персоналізованих рекомендацій для користувача user_id = "196" виключно серед неоцінених ним фільмів. Основним обмеженням такого підходу є проблема «холодного старту» (неможливість сформувати персоналізовані рекомендації для нових користувачів або нових фільмів, відсутніх у trainset), а також залежність якості прогнозів від розрідженості матриці оцінок.   
---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [ ]:
import numpy as np
import pandas as pd

class CustomUserBasedCF:
    def __init__(self, k=30):
        self.k = k
        self.user_item_matrix = None
        self.user_means = None
        self.global_mean = None
        self.sim_matrix = None
        self.user_to_idx = {}
        self.item_to_idx = {}
        self.idx_to_item = {}

    def fit(self, df):
        self.global_mean = df['rating'].mean()

        pivot = df.pivot(index='user_id', columns='item_id', values='rating')
        self.user_to_idx = {uid: idx for idx, uid in enumerate(pivot.index)}
        self.item_to_idx = {iid: idx for idx, iid in enumerate(pivot.columns)}
        self.idx_to_item = {idx: iid for iid, idx in self.item_to_idx.items()}

        R = pivot.to_numpy(dtype=np.float64)
        self.user_item_matrix = R

        self.user_means = np.nanmean(R, axis=1)

        R_centered = R - self.user_means[:, np.newaxis]
        R_filled = np.nan_to_num(R_centered, nan=0.0)

        norms = np.linalg.norm(R_filled, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        R_norm = R_filled / norms
        self.sim_matrix = R_norm @ R_norm.T
        np.fill_diagonal(self.sim_matrix, 0.0)

        self.R_centered_filled = R_filled
        return self

    def predict_one(self, user_id, item_id):

        if user_id not in self.user_to_idx:
            return self.global_mean
        u_idx = self.user_to_idx[user_id]
        if item_id not in self.item_to_idx:
            return self.user_means[u_idx]

        i_idx = self.item_to_idx[item_id]

        rated_mask = ~np.isnan(self.user_item_matrix[:, i_idx])
        if not np.any(rated_mask):
            return self.user_means[u_idx]

        sims = self.sim_matrix[u_idx, rated_mask]
        diffs = self.R_centered_filled[rated_mask, i_idx]

        pos_mask = sims > 0
        if not np.any(pos_mask):
            return self.user_means[u_idx]

        sims = sims[pos_mask]
        diffs = diffs[pos_mask]

        if len(sims) > self.k:
            top_k_idx = np.argpartition(sims, -self.k)[-self.k:]
            sims = sims[top_k_idx]
            diffs = diffs[top_k_idx]

        pred = self.user_means[u_idx] + np.dot(sims, diffs) / np.sum(np.abs(sims))
        return float(np.clip(pred, 1.0, 5.0))

    def evaluate(self, test_df):
        preds = [
            self.predict_one(u, i)
            for u, i in zip(test_df['user_id'], test_df['item_id'])
        ]
        y_true = test_df['rating'].to_numpy(dtype=np.float64)
        y_pred = np.array(preds, dtype=np.float64)

        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        mae = np.mean(np.abs(y_true - y_pred))
        return rmse, mae

    def recommend_top_n(self, user_id, n=10):
        if user_id not in self.user_to_idx:
            return pd.DataFrame(columns=['item_id', 'predicted_rating'])

        u_idx = self.user_to_idx[user_id]
        unrated_indices = np.where(np.isnan(self.user_item_matrix[u_idx]))[0]

        recs = []
        for i_idx in unrated_indices:
            raw_iid = self.idx_to_item[i_idx]
            score = self.predict_one(user_id, raw_iid)
            recs.append((raw_iid, score))

        return (
            pd.DataFrame(recs, columns=['item_id', 'predicted_rating'])
            .sort_values(by='predicted_rating', ascending=False)
            .head(n)
            .reset_index(drop=True)
        )

custom_cf = CustomUserBasedCF(k=30)
custom_cf.fit(train_df)

custom_rmse, custom_mae = custom_cf.evaluate(test_df)
print(f"CustomUserBasedCF -> Test RMSE: {custom_rmse:.4f}, Test MAE: {custom_mae:.4f}")

custom_top10 = custom_cf.recommend_top_n("196", n=10)
print("\nТоп-10 рекомендацій від власного алгоритму для user_id = 196:")
display(custom_top10)

CustomUserBasedCF -> Test RMSE: 0.9337, Test MAE: 0.7280

Топ-10 рекомендацій від власного алгоритму для user_id = 196:


,item_id,predicted_rating
0,1628,5.000000
1,1629,5.000000
2,1642,5.000000
3,1500,5.000000
4,1554,5.000000
5,1306,5.000000
6,1103,5.000000
7,1398,4.980296
8,1467,4.957341
9,1653,4.871921
